# 01 — Exploração de Dados

**MedAI Triage** · Fase 2 do desafio

Este notebook caracteriza a base **Breast Cancer Wisconsin (Diagnostic)** antes de qualquer modelagem: estrutura, estatísticas descritivas, distribuições por classe, valores atípicos e separabilidade individual das features.

> **Toda a lógica vive em `src/medai/`.** Este notebook importa, executa e narra — ele não define funções. Isso garante que o que é analisado aqui é exatamente o que roda na API.

**O que cada seção responde:**

| Seção | Pergunta |
|---|---|
| 1 | Como a base está estruturada? |
| 2 | Como o alvo se distribui, e isso exige tratamento? |
| 3 | Quais são as estatísticas de cada feature? |
| 4 | Como cada feature se distribui em cada classe? |
| 5 | Quanta informação cada feature carrega isoladamente? |
| 6 | Há valores atípicos, e o que fazer com eles? |
| 7 | As features são redundantes entre si? |
| 8 | As classes são separáveis? |

In [ ]:
import sys
import warnings
from pathlib import Path

import pandas as pd

# Raiz do projeto no path: permite importar `medai` sem instalar em modo editável.
RAIZ = Path.cwd().parent
if str(RAIZ / "src") not in sys.path:
    sys.path.insert(0, str(RAIZ / "src"))

from medai import viz
from medai.config import carregar_config
from medai.data import explore
from medai.data.loader import carregar_dados_brutos
from medai.data.validate import validar_dataset

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

viz.configurar_estilo(interativo=True)

config = carregar_config(RAIZ / "configs" / "breast_cancer.yaml")
print(f"Configuração: {config.dataset.nome} (hash {config.hash_config})")
print(f"Classe positiva: {config.dataset.valor_positivo}")

---
## 1. Carregamento e estrutura

O carregador tenta o CSV do Kaggle e cai para o `scikit-learn` quando ele não existe (ADR-002). As duas rotas produzem o mesmo schema canônico.

In [ ]:
df = carregar_dados_brutos(config)
resultado = validar_dataset(df, config)
print(resultado)

In [ ]:
df.head()

In [ ]:
estrutura = explore.resumo_estrutural(df)
for chave, valor in estrutura.items():
    print(f"{chave:>20}: {valor}")

In [ ]:
df.info()

### Leitura

A base é pequena e limpa: **569 linhas × 31 colunas**, 30 features `float64` e um alvo categórico. **Nenhum valor ausente e nenhuma duplicata.**

Isso tem duas consequências opostas que convém separar:

- **A favor:** não há decisão difícil de imputação a tomar aqui, e os resultados não ficam contaminados por escolhas de preenchimento.
- **Contra:** 569 amostras é pouco. O conjunto de teste terá ~114 linhas, em que **cada erro vale 0,9 ponto percentual**. Diferenças pequenas de desempenho entre modelos serão ruído, não sinal — e é por isso que a Fase 6 reporta intervalos sobre múltiplas sementes em vez de números pontuais.

Ainda assim, o pipeline implementará imputação (Fase 3). Não por este dataset precisar, mas porque a API receberá payloads incompletos em produção e porque o mesmo pipeline roda sobre o Pima Diabetes, onde ausentes aparecem disfarçados de zero.

---
## 2. Distribuição do alvo

In [ ]:
classes = explore.distribuicao_classes(df, config)
classes

In [ ]:
fig = viz.plot_distribuicao_classes(df, config)

### Leitura — e a decisão de métrica

**212 malignos (37,3%)** contra **357 benignos (62,7%)**. Desbalanceamento **moderado**.

A linha tracejada é o ponto mais importante deste notebook: um classificador que responde **sempre "benigno"**, sem olhar para nenhuma feature, acerta **62,7%** dos casos. Qualquer modelo que reporte accuracy de 90% precisa ser comparado contra esse piso, não contra zero.

Pior: esse classificador trivial tem **recall zero na classe maligna** — ele deixa passar todos os 212 cânceres. Accuracy alta, utilidade clínica negativa. É exatamente o cenário que a métrica precisa penalizar, e accuracy não penaliza.

**Decisão:** recall da classe maligna é a métrica primária (ADR-003).

**Sobre reamostragem:** 37,3% não é desbalanceamento severo. SMOTE ou undersampling aqui seriam técnica aplicada por reflexo, não por necessidade — e introduziriam amostras sintéticas de uma condição clínica real, com o risco de o modelo aprender artefatos de interpolação. O que o desbalanceamento moderado exige é **estratificação nos splits** e, como variante a testar, `class_weight='balanced'`. Nada além disso.

---
## 3. Estatísticas descritivas

Além do `describe()` padrão, calculamos **assimetria** e **curtose** — que não aparecem nele, mas sustentam a discussão sobre escalonamento.

In [ ]:
features = [c for c in df.columns if c not in {config.dataset.coluna_alvo, *config.dataset.colunas_descartar}]
print(f"{len(features)} features utilizáveis")

descritivas = explore.estatisticas_descritivas(df, features)
descritivas.round(3)

In [ ]:
# As features mais assimétricas — onde o escalonamento padrão não resolve.
descritivas[["mean", "std", "skew", "kurtosis", "cv"]].nlargest(8, "skew").round(2)

### Leitura

Dois pontos relevantes para a Fase 3:

**1. Escalas radicalmente diferentes.** `area_worst` chega a 4.254 enquanto `fractal_dimension_se` fica na casa de 0,003 — mais de seis ordens de magnitude entre features. KNN calcula distância euclidiana: sem padronização, `area_worst` sozinha determinaria toda a vizinhança e as outras 29 features seriam decorativas. A regressão logística sofre de forma parecida na convergência. **`StandardScaler` é obrigatório**, não opcional.

**2. Assimetria forte nas features `_se`.** `area_se` tem skew 5,45 e curtose muito alta; `concavity_se`, 5,11. Cauda longa à direita.

Aqui cabe uma ressalva contra o reflexo: **padronizar não corrige assimetria**. O `StandardScaler` centra e redimensiona, mas uma distribuição com skew 5 continua com skew 5 depois dele. Transformação logarítmica corrigiria — mas só beneficiaria os modelos lineares, enquanto árvores e boosting são completamente indiferentes a transformações monotônicas. Fica registrado como **hipótese a testar na Fase 4**, não como etapa a aplicar sem evidência.

---
## 4. Distribuições por classe

In [ ]:
fig = viz.plot_grade_distribuicoes(df, config, features, titulo="Distribuição de cada feature por diagnóstico")

### Leitura

A grade revela três padrões:

1. **Separação visível** em `concave points`, `perimeter`, `radius`, `area` e `concavity` — as curvas laranja (maligno) e azul (benigno) têm modas claramente distintas, com sobreposição parcial.
2. **Sobreposição quase total** em `smoothness_se`, `symmetry_se`, `texture_se` e `fractal_dimension_mean` — as duas curvas praticamente coincidem. Essas features não separam nada isoladamente.
3. **Caudas longas à direita** nas features `_se`, confirmando visualmente a assimetria da seção anterior.

Note que a sobreposição parcial nas melhores features já antecipa um achado da Fase 6: **vai existir uma região de incerteza**. Nenhuma feature separa perfeitamente, então haverá casos em que o modelo não terá base para decidir — e é para eles que a zona cinzenta existe.

In [ ]:
cohen_d = explore.separabilidade_cohen_d(df, config, features)
top_8 = cohen_d.head(8).index.tolist()

fig = viz.plot_boxplots(df, config, top_8, titulo="Boxplots — 8 features de maior separabilidade")

In [ ]:
features_worst = [f for f in features if f.endswith("_worst")]
fig = viz.plot_violinos(df, config, features_worst[:10], titulo="Features '_worst' — o achado mais atípico de cada lâmina")

### Leitura dos boxplots e violinos

Nas features de maior separabilidade, as caixas das duas classes mal se tocam — mas **as caudas se sobrepõem**. Existem tumores benignos com medidas na faixa dos malignos, e vice-versa. Esses são os casos que vão gerar os erros do modelo, e são justamente os que mais merecem revisão humana.

Os violinos mostram algo clinicamente coerente: nas features `_worst`, a distribuição dos malignos é **mais larga e mais deslocada**. Faz sentido — o sufixo `_worst` é a média dos três núcleos mais extremos da lâmina, e é precisamente neles que o citopatologista se apoia. Uma amostra majoritariamente normal com poucos núcleos muito alterados é suspeita, e a média simples esconderia isso.

---
## 5. Separabilidade individual (d de Cohen)

Diferença das médias entre classes, em unidades de desvio padrão combinado. Mede quanto cada feature separa **isoladamente**, antes de qualquer modelo.

In [ ]:
fig = viz.plot_separabilidade(cohen_d)

In [ ]:
print("Maior separabilidade:")
print(cohen_d.head(8).round(3).to_string())
print("\nSem separabilidade (|d| < 0,2):")
print(cohen_d[cohen_d.abs() < 0.2].round(3).to_string())

### Leitura

**`concave points_worst` lidera com d = 2,60** — efeito enorme (a convenção trata 0,8 como "grande"). As seis primeiras posições são todas de `concave points`, `perimeter` ou `radius`, nas variantes `_worst` e `_mean`.

**Dois achados que orientam as fases seguintes:**

1. **As features `_worst` dominam o topo.** Não é coincidência estatística: reflete como o diagnóstico citopatológico funciona na prática.

2. **Cinco features não separam nada** (|d| < 0,2): `smoothness_se`, `fractal_dimension_mean`, `texture_se`, `symmetry_se` e `fractal_dimension_se`.

O segundo ponto é uma **expectativa falsificável** para a Fase 6. Se o feature importance ou o SHAP atribuírem peso alto a `smoothness_se`, há duas possibilidades: ou a feature tem valor em interação com outras (o que o d de Cohen, sendo univariado, não capta), ou o modelo está aprendendo ruído. Registrar a expectativa agora é o que permite distinguir os dois casos depois — sem isso, qualquer resultado pareceria plausível retrospectivamente.

---
## 6. Valores atípicos

In [ ]:
outliers = explore.detectar_outliers(df, features)
outliers.head(10)

In [ ]:
por_classe = explore.proporcao_outliers_por_classe(df, config, features)
por_classe

In [ ]:
fig = viz.plot_outliers_por_classe(por_classe, config)

### Leitura — e a decisão de manter

`area_se` tem 65 valores fora da cerca IQR (11,4% das amostras), seguida de `radius_se` e `perimeter_se` com 38 cada. Pelo critério padrão, haveria bastante coisa a "limpar".

**Não vamos remover nada**, e o gráfico acima é a razão:

| Classe | Amostras atípicas |
|---|---|
| Benigno | **16,0%** |
| Maligno | **53,8%** |

Mais da metade dos casos malignos tem ao menos uma feature fora da cerca IQR, contra um sexto dos benignos. **Os outliers não são ruído: são a própria classe de interesse.**

Isso é esperado clinicamente. Um tumor maligno é, por definição, um tecido com crescimento desregulado — núcleos maiores, contornos mais irregulares, maior heterogeneidade. "Fora do padrão" é a descrição da patologia, não um defeito de medição.

Remover outliers por regra estatística aqui **descartaria preferencialmente os casos que o modelo mais precisa aprender a identificar**, e o efeito seria perverso: as métricas de teste melhorariam (o problema ficaria artificialmente mais fácil) enquanto a utilidade clínica real cairia. É o tipo de armadilha em que um pipeline automatizado cai sem avisar.

**Decisão (ADR-007): manter todos os valores atípicos**, documentando a escolha.

A única checagem de corrupção que faz sentido é por valores **fisicamente impossíveis** — medidas geométricas negativas, por exemplo. A validação cobre isso e não encontrou nenhum.

---
## 7. Correlação entre features

In [ ]:
fig = viz.plot_heatmap_correlacao(df, features, metodo="pearson")

In [ ]:
# Pares mais correlacionados — candidatos a redundância.
corr = df[features].corr().abs()
import numpy as np
pares = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    .stack()
    .sort_values(ascending=False)
)
pares.head(12).round(4).to_frame("|r|")

### Leitura

Os blocos vermelho-escuros não são achado surpreendente: são **consequência geométrica**. `perimeter ≈ 2πr` e `area ≈ πr²`, de modo que `radius`, `perimeter` e `area` medem a mesma grandeza em unidades diferentes. A correlação acima de 0,99 entre eles é estrutural, não empírica.

O trio se repete nos três sufixos (`_mean`, `_se`, `_worst`), gerando **nove features que carregam essencialmente três informações**.

**Por que isso importa, e por que não é um problema de acurácia:** modelos de árvore lidam bem com features redundantes — escolhem uma, ignoram as outras, e a acurácia não sofre. O dano é na **interpretação**:

- na regressão logística, os coeficientes ficam instáveis e podem até inverter de sinal;
- no SHAP, o crédito se divide entre as features correlacionadas, fazendo **cada uma parecer menos importante do que o grupo realmente é**.

É por isso que a Fase 4 compara dois conjuntos de features — as 30 completas e um subconjunto sem os redundantes (ADR-009). A diferença entre as duas explicações é um resultado em si.

A análise formal, com Spearman e VIF, é a Fase 3.

---
## 8. Separabilidade em conjunto (PCA)

In [ ]:
fig = viz.plot_pca_2d(df, config, features)

### Leitura

Duas componentes capturam **63,2% da variância** (PC1 sozinha, 44,3%), e a separação ao longo de PC1 é nítida: benignos concentrados à esquerda, malignos espalhados à direita.

Três observações:

1. **O problema é aprendível.** A separação visível em apenas duas dimensões, sem supervisão nenhuma, indica que modelos lineares já devem ir bem. Isso calibra a expectativa: accuracy alta aqui não é mérito do algoritmo, é característica do dataset.

2. **Existe uma faixa de sobreposição** em torno de PC1 ≈ 0 a 2, onde as duas classes se misturam. **É ali que moram os erros do modelo** — e é a justificativa empírica para a zona cinzenta: nessa região, nenhuma informação disponível basta para decidir, e forçar um rótulo binário seria fingir uma certeza que os dados não sustentam.

3. **A dispersão dos malignos é muito maior.** Benignos formam um aglomerado compacto; malignos se espalham. Coerente com a análise de outliers — "normal" é um estado estreito, "patológico" tem muitas formas.

> **Nota metodológica:** este PCA foi ajustado sobre o dataset inteiro. Isso é aceitável **porque nenhum modelo está sendo treinado aqui** — o objetivo é enxergar a estrutura dos dados. Se as componentes principais fossem usadas como features de um modelo, o PCA teria de ser ajustado **apenas no treino**, sob pena de vazamento. A distinção é sutil e vale registrar.

---
## 9. Síntese

### O que aprendemos

| Achado | Consequência |
|---|---|
| 569 × 31, sem nulos nem duplicatas | Teste de ~114 amostras: diferenças pequenas entre modelos serão ruído |
| 37,3% malignos | Estratificar os splits; **não** aplicar reamostragem |
| Classificador trivial acerta 62,7% | Accuracy é inadequada como métrica de decisão |
| Escalas variam em 6 ordens de magnitude | `StandardScaler` obrigatório para KNN e regressão logística |
| `area_se` com skew 5,45 | Transformação log é hipótese a testar, não etapa a aplicar |
| `concave points_worst` com d = 2,60 | Expectativa contra a qual validar o SHAP na Fase 6 |
| 5 features com \|d\| < 0,2 | Se o modelo valorizá-las, investigar |
| 53,8% dos malignos são atípicos vs 16,0% dos benignos | **Manter outliers** — são o sinal, não ruído |
| `radius`/`perimeter`/`area` com r > 0,99 | Comparar feature set completo vs reduzido (ADR-009) |
| PCA separa em 2D, com faixa de sobreposição | Problema aprendível; zona cinzenta tem base empírica |

### Decisões que seguem para a Fase 3

1. Imputação mediana no pipeline, **apesar** de não haver ausentes — para a API e para o Pima.
2. `StandardScaler` em todas as features numéricas.
3. **Nenhuma remoção de outliers.**
4. Split estratificado 60/20/20, com todo `fit` restrito ao treino.
5. Análise formal de correlação com Spearman e VIF, e construção do feature set reduzido.

### O que ainda não sabemos

- Se a transformação logarítmica melhora os modelos lineares.
- Se o feature set reduzido perde acurácia, ou só ganha interpretabilidade.
- Quão calibradas serão as probabilidades — determinante para a UI poder exibir "87% de probabilidade" honestamente.